# Collision checking

As part of this exercise, you will implement a collision checker for a robot navigating a 2D environment with static obstacles.

**Note: This is a code-only exercise — you don't need the Duckiebot.**

You will be working in the [`collision_checker.py`][file] file.

[file]: ../../packages/collision_checker.py

## Data structures and protocol

The data structures are defined in the [`collision_protocol.py`][file] file in this repository.

We **strongly** suggest opening the [`collision_protocol.py`][file] link in a separate window, and cross-referencing the information given here with the code definitions.

[file]: ../../packages/collision_protocol.py

The input to the collision checker is a `MapDefinition`, which contains the `environment` (all obstacles the robot can collide with) and the `body` (all parts that make up the robot). Both are lists of `PlacedPrimitive`s.

A `PlacedPrimitive` combines a `FriendlyPose` with a `Primitive` (either a `Circle` or a `Rectangle`):

```python
@dataclass
class PlacedPrimitive:
    pose: FriendlyPose
    primitive: Primitive
    appearance: Optional[Appearance] = None
```

```python
@dataclass
class FriendlyPose:
    x: float
    y: float
    theta_deg: float    # 0° = +x axis, counter-clockwise
```

`theta_deg` starts at zero in the positive x-axis direction and increases counter-clockwise.

A `Primitive` is either a `Circle` or a `Rectangle`. A `Circle` is defined by its `radius`. A `Rectangle` is defined by four offsets relative to its own pose:

- `xmax` / `xmin`: distance to the right / left side in the primitive's local x-axis.
- `ymax` / `ymin`: distance to the top / bottom side in the primitive's local y-axis.

```python
@dataclass
class Circle:
    radius: float

@dataclass
class Rectangle:
    xmin: float
    ymin: float
    xmax: float
    ymax: float

Primitive = Union[Circle, Rectangle]
```

All offsets are in the primitive's own coordinate frame, so `theta_deg` in the pose rotates the entire shape in the world.

The checker receives a `CollisionCheckQuery` with a new robot pose and must return a `CollisionCheckResult` (a boolean: `True` = collision, `False` = free).

## Implementation

In [`collision_checker.py`][file] you will find the `CSpaceChecker` class to implement. The main interface is:

[file]: ../../packages/collision_checker.py

```python
class CSpaceChecker:
    def __init__(self, environment: List[PlacedPrimitive], robot_body: List[PlacedPrimitive]):
        ...

    def check(self, robot_pose: FriendlyPose) -> bool:
        """Return True if the robot at robot_pose collides with any obstacle."""
        ...

    def distance(self, robot_pose: FriendlyPose) -> float:
        """Distance from robot_pose to the nearest obstacle (0 if in collision)."""
        ...
```

A convenience wrapper `check_collision(environment, body, pose)` is also provided for one-off queries.

## Visualization

The cell below samples random robot poses and colours them by collision status.

Before testing your solution, the environment looks like this (blue = free, red = collision):

![query](../../assets/images/env18.png)

After running your checker, four colours show whether you were right or wrong:

- $\color{blue}{\text{Blue}}$: not in collision, correctly identified.
- $\color{orange}{\text{Orange}}$: not in collision, but you said it was (**false positive**).
- $\color{red}{\text{Red}}$: in collision, correctly identified.
- $\color{pink}{\text{Pink}}$: in collision, but you missed it (**false negative**).

![result](../../assets/images/env18-result.png)

## Tips: think in configuration space

### What is configuration space?

The **configuration** of the robot is everything you need to fully describe its pose: $(x, y, \theta)$.

The **configuration space** (C-space) is the 3D space of all configurations. A configuration is *in collision* if the robot placed at that configuration overlaps any obstacle. The set of all colliding configurations is called the **C-space obstacle**.

The key insight is that **for a fixed orientation $\theta$**, the C-space obstacle is just a 2D region in the $(x, y)$ plane. Checking whether the robot collides at $(x, y, \theta)$ then reduces to a fast **point-in-region** query:

    collision(x, y, θ) = (x, y) ∈ forbidden_region(θ)

### Computing the forbidden region via Minkowski sums

For a robot body part $B$ and an obstacle $O$, the forbidden region at orientation $\theta$ is:

$$\mathcal{C}O(\theta) = O \oplus (-B_\theta)$$

where $B_\theta$ is the body part shape rotated to orientation $\theta$, $-B_\theta$ is its reflection through the origin, and $\oplus$ denotes the **Minkowski sum**.

Intuitively: $\mathcal{C}O(\theta)$ is the set of robot-centre positions at which any part of the body (at orientation $\theta$) would touch $O$.

For two convex polygons $A$ and $B$, the Minkowski sum $A \oplus B$ is the convex hull of all pairwise vertex sums — equivalently, it is the union of $A$ translated to every vertex of $B$:

$$A \oplus B = \text{convex\_hull}\bigl(\bigcup_{v \in B} (A + v)\bigr)$$

The special case where one shape is a circle of radius $r$ is even cleaner:

$$A \oplus \text{disk}(r) = A.\text{buffer}(r)$$

which is just a dilation — Shapely can compute this exactly.

### Lazy caching

Computing the Minkowski sum for a slice is moderately expensive, but it only needs to happen **once per orientation**. After that, all $(x, y)$ queries at the same $\theta$ are free point-in-polygon tests.

Store the computed forbidden region in a dict keyed by $\theta$ and compute it on first use:

```python
def _slice(self, theta_deg):
    if theta_deg not in self._cache:
        self._cache[theta_deg] = self._compute_slice(theta_deg)
    return self._cache[theta_deg]
```

This pays off whenever multiple poses share the same orientation — which is exactly what happens during motion planning, where the planner queries a grid of $(x, y)$ positions at each of a fixed set of orientations.

## Tips: putting it together

### Body shape at a given orientation

To build the C-space slice for orientation $\theta$, you first need the shape of each body part when the **robot centre sits at the origin** and the robot faces $\theta$.

For a body part with local pose $(dx, dy, d\theta)$ and primitive shape $P$:

1. Apply the local pose: rotate $P$ by $d\theta$, then translate by $(dx, dy)$.
2. Apply the global orientation: rotate the result by $\theta$ about the origin.

In Shapely:

```python
shape = rotate(primitive_shape, body_part.pose.theta_deg, origin=(0, 0))
shape = translate(shape, body_part.pose.x, body_part.pose.y)
shape = rotate(shape, theta_deg, origin=(0, 0))   # global orientation
```

### Reflecting and summing

The C-space obstacle is $O \oplus (-B_\theta)$. To reflect $B_\theta$ through the origin:

```python
body_reflected = scale(body_shape, xfact=-1, yfact=-1, origin=(0, 0))
```

Then:

- **Obstacle is a `Circle`** (radius $r$, centre $(cx, cy)$):  
  $\mathcal{C}O = $ `translate(body_reflected.buffer(r), cx, cy)` — exact, using Shapely's dilation.

- **Obstacle is a `Rectangle`** (convex polygon):  
  $\mathcal{C}O = $ convex hull of the obstacle translated to each vertex of the reflected body:

```python
coords = list(body_reflected.exterior.coords[:-1])
parts  = [translate(obs_shape, dx, dy) for dx, dy in coords]
cspace_obstacle = unary_union(parts).convex_hull
```

### Combining all pairs

The full forbidden region at $\theta$ is the union over all (body part, obstacle) pairs:

```python
parts = []
for body_part in robot_body:
    body_shape    = _body_at_theta(body_part, theta_deg)
    body_reflected = scale(body_shape, xfact=-1, yfact=-1, origin=(0, 0))
    for obs in environment:
        parts.append(_cspace_obstacle(obs, body_reflected))
return unary_union(parts)
```

### Querying

Once the forbidden region is built, a collision check is just:

```python
forbidden.disjoint(Point(x, y))   # True → no collision
```

And clearance distance is:

```python
forbidden.distance(Point(x, y))   # 0 if in collision
```

## Testing your code

Finally you can test your code by running the following cells, which will give you a score. 

In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

from matplotlib import pyplot as plt

In [2]:
from make_environments import sample_environment, sample_body, sample_pose, get_ground_truth2, COLOR_BG, COLOR_COLLISION, COLOR_NOCOLLISION, COLOR_COLLISION_WRONG, COLOR_NOCOLLISION_WRONG
from collision_drawing import plot_geometry
from collision_protocol import MapDefinition, CollisionCheckQuery, CollisionCheckResult, Rectangle
from se2_utils import SE2, pose_from_friendly
from collision_checker import CSpaceChecker

W = H = 5
area = Rectangle(xmin=0, ymin=0, xmax=W, ymax=H)
num_samples = 20

environment = sample_environment(area)
body = sample_body()
params = MapDefinition(body=body, environment=environment)

# Build the checker once — C-space slices are computed lazily and cached per orientation.
checker = CSpaceChecker(environment, body)

COLOR_OBSTACLES = "brown"
f = plt.figure()
plt.tight_layout()
ax: plt.Axes = plt.gca()
ax.add_artist(plt.Rectangle((0, 0), width=W, height=H, color=COLOR_BG, zorder=-10))

plot_geometry(ax, SE2.identity(), environment, COLOR_OBSTACLES, 0)

num_correct = 0

for i in range(num_samples):
    p = sample_pose(area)
    query = CollisionCheckQuery(pose=p)
    gt: CollisionCheckResult = get_ground_truth2(params, query)

    result = checker.check(p)

    s0 = pose_from_friendly(p)
    if gt.collision and result:
        color = COLOR_COLLISION
        num_correct += 1
    elif gt.collision and not result:
        color = COLOR_COLLISION_WRONG
    elif result and not gt.collision:
        color = COLOR_NOCOLLISION_WRONG
    else:
        color = COLOR_NOCOLLISION
        num_correct += 1

    for x in body:
        x.appearance.fillcolor = color
        plot_geometry(ax, s0, body, color, 10)
        ax.set_aspect("equal")
        plt.axis((0, W, 0, H))
        plt.axis("off")

print(f"Your final score was {num_correct/num_samples*100}%")

NameError: name 'pyplot' is not defined